In [4]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import MemorySaver #MemorySaver- is a kind of checkpointer. Jiski help se persistence implement kr skte apne workflow me. Ye hamare sare ke sare state values ko (intermediate+final) values ko memory me save krta hai.. RAM me save krta hai.

In [5]:
load_dotenv()
llm = ChatOpenAI()

In [6]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [8]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content
    
    return {'joke':response}

In [9]:
def generate_explanation(state: JokeState):

    prompt = f'Write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content
    
    return {'explanation':response}

In [10]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = MemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [11]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic': 'technology'}, config=config1)

{'topic': 'technology',
 'joke': 'Why did the computer go to the doctor? Because it had a virus!',
 'explanation': 'This joke plays on the dual meaning of the word "virus." In the context of computers, a virus is a type of malicious software that can infect a computer and cause it to malfunction. In this joke, the computer "went to the doctor" because it had a virus, implying that the computer is sick and needs to be treated for the virus infection. This humorous twist on a common computer problem adds a lighthearted and playful element to the situation.'}

In [12]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'technology', 'joke': 'Why did the computer go to the doctor? Because it had a virus!', 'explanation': 'This joke plays on the dual meaning of the word "virus." In the context of computers, a virus is a type of malicious software that can infect a computer and cause it to malfunction. In this joke, the computer "went to the doctor" because it had a virus, implying that the computer is sick and needs to be treated for the virus infection. This humorous twist on a common computer problem adds a lighthearted and playful element to the situation.'}, next=(), config={'configurable': {'thread_id': '1', 'thread_ts': '1f0f6f51-a07c-60a0-8002-95b782f9afe5'}}, metadata={'source': 'loop', 'step': 2, 'writes': {'generate_explanation': {'explanation': 'This joke plays on the dual meaning of the word "virus." In the context of computers, a virus is a type of malicious software that can infect a computer and cause it to malfunction. In this joke, the computer "went to t

In [14]:
list(workflow.get_state_history(config1)) #to see intermediate state values

[StateSnapshot(values={}, next=('__start__',), config={'configurable': {'thread_id': '1', 'thread_ts': '1f0f6f51-762d-6999-bfff-da7c93a1bb48'}}, metadata={'source': 'input', 'step': -1, 'writes': {'topic': 'technology'}}, created_at='2026-01-21T18:14:56.825999+00:00', parent_config=None),
 StateSnapshot(values={'topic': 'technology'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'thread_ts': '1f0f6f51-7635-6481-8000-1f06db714abc'}}, metadata={'source': 'loop', 'step': 0, 'writes': None}, created_at='2026-01-21T18:14:56.829145+00:00', parent_config=None),
 StateSnapshot(values={'topic': 'technology', 'joke': 'Why did the computer go to the doctor? Because it had a virus!'}, next=('generate_explanation',), config={'configurable': {'thread_id': '1', 'thread_ts': '1f0f6f51-82f5-6cac-8001-e4e0ebbc1132'}}, metadata={'source': 'loop', 'step': 1, 'writes': {'generate_joke': {'joke': 'Why did the computer go to the doctor? Because it had a virus!'}}}, created_at='2026-01-

In [16]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic': 'AI'}, config=config2)

{'topic': 'AI',
 'joke': "Why did the artificial intelligence break up with its computer? Because it couldn't handle all the emotional baggage!",
 'explanation': "This joke plays on the idea of artificial intelligence being unable to handle or process emotions. In this case, the AI broke up with its computer (presumably a human) because it couldn't handle all the emotional baggage that comes with being in a relationship with a human. The humor comes from the absurdity of the situation and the unexpected twist on a typical breakup scenario."}

In [19]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'AI', 'joke': "Why did the artificial intelligence break up with its computer? Because it couldn't handle all the emotional baggage!", 'explanation': "This joke plays on the idea of artificial intelligence being unable to handle or process emotions. In this case, the AI broke up with its computer (presumably a human) because it couldn't handle all the emotional baggage that comes with being in a relationship with a human. The humor comes from the absurdity of the situation and the unexpected twist on a typical breakup scenario."}, next=(), config={'configurable': {'thread_id': '2', 'thread_ts': '1f0f6f5b-4eac-62ed-8002-53b039fc2b9d'}}, metadata={'source': 'loop', 'step': 2, 'writes': {'generate_explanation': {'explanation': "This joke plays on the idea of artificial intelligence being unable to handle or process emotions. In this case, the AI broke up with its computer (presumably a human) because it couldn't handle all the emotional baggage that comes wi

In [20]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={}, next=('__start__',), config={'configurable': {'thread_id': '2', 'thread_ts': '1f0f6f5b-3455-682b-bfff-d51af6e3f68b'}}, metadata={'source': 'input', 'step': -1, 'writes': {'topic': 'AI'}}, created_at='2026-01-21T18:19:18.357201+00:00', parent_config=None),
 StateSnapshot(values={'topic': 'AI'}, next=('generate_joke',), config={'configurable': {'thread_id': '2', 'thread_ts': '1f0f6f5b-3457-6f94-8000-7b90a938bff7'}}, metadata={'source': 'loop', 'step': 0, 'writes': None}, created_at='2026-01-21T18:19:18.358210+00:00', parent_config=None),
 StateSnapshot(values={'topic': 'AI', 'joke': "Why did the artificial intelligence break up with its computer? Because it couldn't handle all the emotional baggage!"}, next=('generate_explanation',), config={'configurable': {'thread_id': '2', 'thread_ts': '1f0f6f5b-3aff-615c-8001-4c07178a14d3'}}, metadata={'source': 'loop', 'step': 1, 'writes': {'generate_joke': {'joke': "Why did the artificial intelligence break up with its com

In [20]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):

    messages: Annotated[list[BaseMessage], add_messages] #Base Message- System message, Human message, AI message, Tools message, Reducer- to maintain complete conversation history.. pichli chat replace nhi hoti nyi chat se.. pichle wala htaga ni, nye waala add hota jayega. List ke andr messages ko append kr denge

In [21]:
llm = ChatOpenAI()

def chat_node(state: ChatState):

    # take user query from state
    messages = state['messages']

    # send to llm
    response = llm.invoke(messages)

    # response store state
    return {'messages': [response]}

In [38]:

checkpointer = MemorySaver()

graph = StateGraph(ChatState)

# add nodes
graph.add_node('chat_node', chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile(checkpointer=checkpointer)

In [28]:
chatbot

CompiledStateGraph(nodes={'__start__': PregelNode(config={'tags': ['langsmith:hidden']}, channels=['__start__'], triggers=['__start__'], writers=[ChannelWrite<messages>(recurse=True, writes=[ChannelWriteEntry(channel='messages', value=<object object at 0x00000246BF126260>, skip_none=False, mapper=_get_state_key(recurse=False))], require_at_least_one_of=['messages']), ChannelWrite<start:chat_node>(recurse=True, writes=[ChannelWriteEntry(channel='start:chat_node', value='__start__', skip_none=False, mapper=None)], require_at_least_one_of=None)]), 'chat_node': PregelNode(config={'tags': []}, channels={'messages': 'messages'}, triggers=['start:chat_node'], mapper=functools.partial(<function _coerce_state at 0x00000246BFC12CA0>, <class '__main__.ChatState'>), writers=[ChannelWrite<chat_node,messages>(recurse=True, writes=[ChannelWriteEntry(channel='chat_node', value='chat_node', skip_none=False, mapper=None), ChannelWriteEntry(channel='messages', value=<object object at 0x00000246BF126260>,

In [15]:
from IPython.display import Image, display

# Visualize the graph
try:
    display(Image(chatbot.get_graph().draw_mermaid_png()))
except Exception:
    # This serves as a fallback if dependencies like pygraphviz or mermaid.ink are unavailable
    print("Could not display the graph visually.")

Could not display the graph visually.


In [30]:
initial_state = {
    'messages': [HumanMessage(content='Speciality of Haryana')]
}

chatbot.invoke(initial_state)

{'messages': [HumanMessage(content='Speciality of Haryana', id='924d7c19-86ed-4f7f-8597-7ed5bd5b7731'),
  AIMessage(content='Haryana is known for its rich cultural heritage, vibrant folk music and dance, and traditional art forms. The state is also famous for its delicious cuisine, particularly its spicy and flavorful vegetarian dishes. Haryana is home to a number of historical and religious sites, including the Kurukshetra battlefield, which is mentioned in the Hindu epic, the Mahabharata. The state is also known for its agricultural prowess, particularly in the production of wheat, rice, and mustard. Additionally, Haryana is home to a number of industrial hubs, particularly in the automotive and manufacturing sectors.', response_metadata={'token_usage': {'completion_tokens': 118, 'prompt_tokens': 12, 'total_tokens': 130, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {

In [32]:
initial_state = {
    'messages': [HumanMessage(content='Speciality of Haryana')]
}

chatbot.invoke(initial_state)['messages'][-1].content

'Haryana is known for its rich cultural heritage and traditional folk music and dances. It is also famous for its vibrant festivals, such as the Holi festival and Teej festival, which are celebrated with great enthusiasm and fervor. Haryana is also known for its delicious cuisine, which includes dishes such as dal makhani, butter chicken, and paneer tikka. Additionally, Haryana is known for its sports culture, particularly in the field of wrestling, with many renowned wrestlers hailing from the state.'

In [39]:
thread_id = '1'

while True:

    user_message = input('Type here: ')

    print('user:', user_message)

    if user_message.strip().lower() in ['exit', 'quit', 'bye']:
        break

    config = {'configurable' : {'thread_id': thread_id}}
    response = chatbot.invoke({'messages': [HumanMessage(content=user_message)]}, config=config)

    print('AI:', response['messages'][-1].content)

user: hi my name is sumit
AI: Hello Sumit, nice to meet you! How can I assist you today?
user: what is my name
AI: Your name is Sumit.
user: can u add 20 and  9
AI: Yes, 20 + 9 equals 29.
user: my sister name is tanisha
AI: That's a lovely name! Is there anything else you would like to share or ask about?
user: can u add 1 to the result of numbers
AI: Sure! If we add 1 to the previous result of 29, we get 30.
user: exit


In [40]:
chatbot.get_state(config=config)

StateSnapshot(values={'messages': [HumanMessage(content='hi my name is sumit', id='190bbcfb-1ff3-452c-b069-b34d995625e2'), AIMessage(content='Hello Sumit, nice to meet you! How can I assist you today?', response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 13, 'total_tokens': 29, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-0d1f31de-dd3b-4150-a985-6ea6d9beac25-0', usage_metadata={'input_tokens': 13, 'output_tokens': 16, 'total_tokens': 29}), HumanMessage(content='what is my name', id='409a18bd-6780-4401-9c27-8fa6be13a1a6'), AIMessage(content='Your name is Sumit.', response_metadata={'token_usage': {'completion_tokens': 6, 'prompt_tokens': 41, 'total_tokens': 47, 'completion_tokens_details': 